# 📈 Módulo 07 - Notebook 01: Resampling y ventanas móviles

## 🕒 Series de Tiempo: Agregaciones temporales y promedios móviles

**Libro:** Saliendo de lo Pandito  
**Módulo:** 07 - Series de Tiempo Financieras  
**Duración estimada:** 70 minutos  
**Dificultad:** 🟡 Intermedio  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** índices de tiempo con DatetimeIndex  
✅ **Aplicar** resampling (cambio de frecuencia)  
✅ **Calcular** ventanas móviles (rolling)  
✅ **Analizar** tendencias con promedios móviles  
✅ **Detectar** estacionalidad en series temporales

---

## 📋 Pre-requisitos

* ✅ Módulo 06 completado (Agregaciones)
* ✅ Conocimiento de fechas en Pandas (pd.to_datetime)
* ✅ Familiaridad con series temporales

---

## 📚 Contenido

1. DatetimeIndex y Series Temporales
2. Resampling: Cambio de Frecuencia
3. Ventanas Móviles (Rolling)
4. Suavizado de Series
5. Detección de Tendencias
6. Caso Integrador: Análisis de Ventas Mensuales

---

## 💡 Por qué importa

**Series de tiempo están en todas partes:**

* 💰 **Finanzas:** Precios de acciones, tipos de cambio
* 📊 **Ventas:** Ingresos mensuales, estacionalidad
* 🏭 **Economía:** PIB, inflación, desempleo
* 🏪 **Retail:** Tráfico, inventario, demanda

**Dominar series de tiempo = Predecir el futuro con datos del pasado**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df_raw = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_raw['fecha'] = pd.to_datetime(df_raw['fecha'])
    
    # Opción 1: Serie temporal agregada (todas las sucursales)
    df_ts = df_raw.groupby('fecha')['ventas'].sum().sort_index()
    df_ts = df_ts.to_frame(name='ventas_totales')
    
    # Opción 2: Serie por sucursal
    df_ts_sucursal = df_raw.pivot_table(
        values='ventas', 
        index='fecha', 
        columns='sucursal_nombre', 
        aggfunc='sum'
    )
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📅 Período: {df_ts.index.min().strftime('%Y-%m-%d')} a {df_ts.index.max().strftime('%Y-%m-%d')}")
    print(f"   📊 Meses totales: {len(df_ts)}")
    print(f"   🏪 Sucursales: {df_raw['sucursal_id'].nunique()}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n📋 DataFrames disponibles:")
    print(f"   • df_ts: Serie temporal agregada (todas las sucursales)")
    print(f"   • df_ts_sucursal: Serie temporal por sucursal")
    print(f"   • df_raw: DataFrame completo con todas las columnas")
    
    print(f"\n🎯 Este notebook usará datos REALES de Los Andes Market")
    print(f"   Los ejemplos trabajarán con series temporales reales")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ts = None
    df_ts_sucursal = None
    df_raw = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Series de Tiempo: Conceptos Fundamentales

### 🕒 ¿Qué es una Serie de Tiempo?

Una **serie de tiempo** es una secuencia de datos ordenados cronológicamente.

**Ejemplo:**
```
Fecha         Ventas
2023-01      100,000
2023-02      125,000
2023-03      115,000
```

**Componentes:**
1. **Índice temporal:** Fechas en orden
2. **Valores:** Métrica que evoluciona en el tiempo

---

### 🗓️ DatetimeIndex

**DatetimeIndex** es el índice especializado de Pandas para series de tiempo.

```python
# Crear DatetimeIndex
fechas = pd.date_range(start='2023-01-01', periods=12, freq='MS')  # Monthly Start
df = pd.DataFrame({'ventas': [...]}, index=fechas)
```

**Ventajas:**
* Operaciones temporales eficientes
* Resample automático
* Slicing por fechas

---

### 🔄 Resampling (Cambio de Frecuencia)

**Resampling** = Cambiar la frecuencia de la serie.

#### 1️⃣ **Downsampling** (Mayor → Menor frecuencia)

```python
# Diario → Mensual
df_mensual = df_diario.resample('M').sum()
```

**Ejemplo:**
```
Diario:         Mensual:
2023-01-01  10  2023-01  310  (suma de 31 días)
2023-01-02  12  2023-02  280  (suma de 28 días)
...
2023-01-31   8
```

---

#### 2️⃣ **Upsampling** (Menor → Mayor frecuencia)

```python
# Mensual → Diario
df_diario = df_mensual.resample('D').ffill()  # forward fill
```

**Métodos de rellenado:**
* `ffill()`: Forward fill (repetir último valor)
* `bfill()`: Backward fill (usar próximo valor)
* `interpolate()`: Interpolación lineal

---

### 📊 Ventanas Móviles (Rolling)

**Rolling window** = Calcular una métrica sobre una ventana deslizante.

```python
# Promedio móvil de 3 meses
df['MA_3M'] = df['ventas'].rolling(window=3).mean()
```

**Visualización:**
```
Mes    Ventas  MA_3M
01     100     NaN      (sólo 1 valor)
02     120     NaN      (sólo 2 valores)
03     110     110.0    (promedio de 100, 120, 110)
04     130     120.0    (promedio de 120, 110, 130)
05     140     126.7    (promedio de 110, 130, 140)
```

---

### 🎯 Funciones de Agregación

**En resample():**
```python
df.resample('M').sum()    # Suma mensual
df.resample('M').mean()   # Promedio mensual
df.resample('M').max()    # Máximo mensual
```

**En rolling():**
```python
df['ventas'].rolling(3).mean()   # Promedio móvil
df['ventas'].rolling(3).std()    # Desviación estándar móvil
df['ventas'].rolling(3).sum()    # Suma móvil
```

---

### 💼 Casos de Uso

| Operación | Uso típico |
|-----------|---------------|
| **Resample diario → mensual** | Reportes mensuales de ventas diarias |
| **Rolling mean (3M)** | Suavizar volatilidad, detectar tendencia |
| **Rolling std** | Medir volatilidad en finanzas |
| **Resample + ffill** | Rellenar datos faltantes |

---

### 💡 Regla de Oro

👉 **Resample** = Cambiar frecuencia  
👉 **Rolling** = Ventana deslizante para suavizar/calcular

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🕒 SERIES DE TIEMPO: RESAMPLING Y ROLLING")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

print("\n🎯 En este notebook aprenderás:")
print("  • DatetimeIndex: Índices temporales")
print("  • Resampling: Cambio de frecuencia (diario → mensual)")
print("  • Rolling: Ventanas móviles (promedios móviles)")
print("  • Suavizado de series temporales")

print("\n📖 Métodos clave:")
print("  - pd.date_range(start, periods, freq)")
print("  - df.resample('M').sum()  # Cambiar frecuencia")
print("  - df['col'].rolling(window=3).mean()  # Ventana móvil")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 📊 Resampling y Rolling con datos reales de Los Andes Market

### 🕒 Nuestra serie temporal real

Los datos de `ventas_mensuales_mendoza_h3` tienen frecuencia **mensual**: una observación por sucursal por mes. Al agregar todas las sucursales, obtenemos una serie temporal consolidada.

```python
# Serie agregada (todas las sucursales)
df_ts = df_raw.groupby('fecha')['ventas'].sum()

# Serie por sucursal (formato ancho)
df_ts_sucursal = df_raw.pivot_table(
    values='ventas', index='fecha', columns='sucursal_nombre', aggfunc='sum'
)
```

---

### 🔄 Resampling aplicado a Los Andes Market

Con datos mensuales podemos hacer:
* **Downsampling a trimestral/anual:** `df_ts.resample('Q').sum()` o `.resample('Y').sum()`
* **NO tiene sentido upsampling** (mensual → diario no agrega información real)

---

### 📊 Rolling aplicado a ventas

```python
# Promedio móvil 3 meses: suaviza fluctuaciones mensuales
df_ts['ma_3m'] = df_ts['ventas'].rolling(3).mean()

# Promedio móvil 12 meses: revela tendencia anual
df_ts['ma_12m'] = df_ts['ventas'].rolling(12).mean()

# Volatilidad móvil: variabilidad mes a mes
df_ts['volatilidad_3m'] = df_ts['ventas'].rolling(3).std()
```

---

### 💡 Preguntas de negocio
* ¿Las ventas muestran tendencia creciente o decreciente?
* ¿Hay estacionalidad (meses consistently altos/bajos)?
* ¿Qué sucursal tiene la mayor volatilidad?
* ¿El promedio móvil de 12 meses revela un cambio de dirección?

In [0]:
import pandas as pd
import numpy as np

print("📊 RESAMPLING Y ROLLING CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df_ts is not None:
    print("\n1️⃣  SERIE TEMPORAL AGREGADA (todas las sucursales)")
    print("-"*70)
    print(f"\n   Período: {df_ts.index.min().strftime('%Y-%m')} a {df_ts.index.max().strftime('%Y-%m')}")
    print(f"   Observaciones: {len(df_ts)} meses")
    print(f"   Ventas promedio: ${df_ts['ventas_totales'].mean():,.0f}")
    print(f"\n   Primeras 5 observaciones:")
    print(df_ts.head())

    print("\n" + "="*70)
    print("\n2️⃣  RESAMPLING: Mensual → Trimestral")
    print("-"*70)

    ventas_trimestral = df_ts['ventas_totales'].resample('QE').sum()
    print("\n   Ventas trimestrales (sum):")
    print(ventas_trimestral.round(0))

    print("\n" + "="*70)
    print("\n3️⃣  RESAMPLING: Mensual → Anual")
    print("-"*70)

    ventas_anual = df_ts['ventas_totales'].resample('YE').agg(['sum', 'mean', 'count'])
    print("\n   Ventas anuales:")
    print(ventas_anual.round(0))

    print("\n" + "="*70)
    print("\n4️⃣  ROLLING: Promedio móvil 3 y 12 meses")
    print("-"*70)

    df_ts['ma_3m'] = df_ts['ventas_totales'].rolling(3).mean()
    df_ts['ma_12m'] = df_ts['ventas_totales'].rolling(12).mean()
    print("\n   Serie con promedios móviles:")
    print(df_ts.round(0).head(15))
    print("\n   💡 ma_3m suaviza fluctuaciones, ma_12m revela tendencia anual")

    print("\n" + "="*70)
    print("\n5️⃣  ROLLING: Volatilidad móvil (desviación)")
    print("-"*70)

    df_ts['volatilidad_3m'] = df_ts['ventas_totales'].rolling(3).std()
    print("\n   Volatilidad móvil (3 meses):")
    print(df_ts[['ventas_totales', 'volatilidad_3m']].round(0).head(15))
    print("\n   💡 Volatilidad alta = ventas inestables, baja = consistentes")

    print("\n" + "="*70)
    print("\n6️⃣  ANÁLISIS POR SUCURSAL: Promedio móvil")
    print("-"*70)

    if df_ts_sucursal is not None:
        ma_sucursal = df_ts_sucursal.rolling(12).mean()
        print("\n   Promedio móvil 12 meses por sucursal (últimas 6 obs):")
        print(ma_sucursal.round(0).tail(6))
        print("\n   💡 Compara la tendencia de cada sucursal en el último año")

    print("\n" + "="*70)
    print("\n7️⃣  DETECCIÓN DE TENDENCIA")
    print("-"*70)

    # Comparar primero vs último promedio móvil de 12 meses
    ma_valid = df_ts['ma_12m'].dropna()
    if len(ma_valid) >= 2:
        primero = ma_valid.iloc[0]
        ultimo = ma_valid.iloc[-1]
        cambio_pct = (ultimo / primero - 1) * 100
        print(f"\n   MA 12m inicial: ${primero:,.0f}")
        print(f"   MA 12m final:   ${ultimo:,.0f}")
        print(f"   Cambio: {cambio_pct:+.1f}%")
        if cambio_pct > 5:
            print("   📈 Tendencia CRECIENTE")
        elif cambio_pct < -5:
            print("   📉 Tendencia DECRECIENTE")
        else:
            print("   ➡️  Tendencia ESTABLE")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

## 🎓 Conclusiones del notebook 07_01

### ✅ Lo que aprendiste

1. **DatetimeIndex:**
   - `pd.date_range(start, periods, freq)` crea índices temporales
   - Permite slicing por fechas: `df['2023-01':'2023-06']`
   - Base para todas las operaciones de series de tiempo

2. **Resampling (cambio de frecuencia):**
   - **Downsampling:** `df.resample('M').sum()` (diario → mensual)
   - **Upsampling:** `df.resample('D').ffill()` (mensual → diario)
   - Frecuencias comunes: `'D'` diario, `'W'` semanal, `'M'` mensual, `'Q'` trimestral, `'Y'` anual

3. **Ventanas móviles (rolling):**
   - `df['col'].rolling(window=3).mean()` — promedio móvil
   - `df['col'].rolling(window=3).std()` — desviación móvil (volatilidad)
   - Los primeros `window-1` valores son `NaN` por definición

4. **Suavizado de series:**
   - Rolling mean elimina ruido y revela la tendencia subyacente
   - Ventanas más grandes = más suave, pero más rezago
   - Útil para detectar cambios de dirección en la serie

5. **Detección de tendencias:**
   - Comparar serie original vs promedio móvil
   - `rolling().mean()` para tendencia, `rolling().std()` para volatilidad
   - Estacionalidad: patrones que se repiten cada N períodos

---

### 🎯 Reglas de Oro

👉 **Regla #1: Ordenar el índice antes de resample/rolling**
```python
# MALO: datos desordenados, resultados incorrectos
df.resample('M').sum()

# BUENO: asegurar orden cronológico
df = df.sort_index()
df.resample('M').sum()
```

👉 **Regla #2: Elegir aggfunc según contexto**
```python
# Ventas acumulativas → sum
df.resample('Q').sum()

# Precios promedio → mean
df.resample('W').mean()

# Último valor conocido → last
df.resample('D').last()
```

👉 **Regla #3: Cuidado con los NaN en rolling**
```python
# Los primeros window-1 valores son NaN
# Usar min_periods=1 para evitarlo (menos preciso al inicio)
df['col'].rolling(window=3, min_periods=1).mean()

# O eliminarlos del análisis
df['col'].rolling(window=3).mean().dropna()
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Diario → mensual (totales) | `df.resample('M').sum()` |
| Mensual → diario (rellenar) | `df.resample('D').ffill()` |
| Suavizar ruido de una serie | `df['col'].rolling(3).mean()` |
| Medir volatilidad | `df['col'].rolling(12).std()` |
| Tendencia de largo plazo | `df['col'].rolling(12).mean()` |
| Tendencia de corto plazo | `df['col'].rolling(3).mean()` |
| Interpolar valores faltantes | `df.resample('D').interpolate()` |
| Reporte trimestral | `df.resample('Q').agg(['sum', 'mean'])` |
| Detectar estacionalidad | Comparar rolling mean vs serie original |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>📈 ¡Resampling y ventanas móviles dominados!</h3>
  <p><i>"El promedio móvil revela lo que el ruido esconde: la tendencia real del negocio."</i></p>
</div>